In [ ]:
import pandas as pd
import numpy as np
import re

archivo = "plan_de_compras_2025.xlsx"

df = pd.read_excel(archivo)


In [ ]:
# Buscar automáticamente las columnas

col_codigo = None

for c in df.columns:
    if "código" in c.lower() and "presup" in c.lower():
        col_codigo = c
        break

if col_codigo is None:
    raise Exception("No se encontró la columna Código presupuestario.")

columnas_monto = [
    c
    for c in df.columns
    if "monto" in c.lower() and "arrastre" in c.lower()
]

print("Código:", col_codigo)
print("Monto:", columnas_monto)

In [ ]:
# no rompe codigos de tipo 22-01-005
def obtener_codigos(valor):

    if pd.isna(valor):
        return []

    texto = str(valor).strip()

    # Caso 1: separados por " - "
    if "-" in texto:
        return [x.strip() for x in texto.split(" - ") if x.strip()]

    # Caso 2: separados por varios espacios
    patron = r'([0-9]{2}-[0-9]{2}-[0-9]{3})'
    encontrados = re.findall(patron, texto)

    if len(encontrados) > 1:
        return encontrados

    return [texto]

In [ ]:
# Crear un nuevo DataFrame con los códigos duplicados
filas = []

for _, fila in df.iterrows():

    codigos = obtener_codigos(fila[col_codigo])

    if len(codigos) == 0:
        filas.append(fila.copy())
        continue

    if len(codigos) == 1:
        nueva = fila.copy()
        nueva[col_codigo] = codigos[0]
        filas.append(nueva)
        continue

    # Primera fila conserva el monto
    primera = fila.copy()
    primera[col_codigo] = codigos[0]
    filas.append(primera)

    # Filas nuevas
    for codigo in codigos[1:]:

        nueva = fila.copy()
        nueva[col_codigo] = codigo

        # Borrar completamente el monto
        for c in columnas_monto:
            nueva[c] = np.nan

        filas.append(nueva)

nuevo_df = pd.DataFrame(filas)

In [ ]:
# verificacion
display(
    nuevo_df[
        [col_codigo] + columnas_monto
    ].head(30)
)

In [ ]:
# salida a archivo
salida = "plan_de_compras_nuevo.xlsx"

with pd.ExcelWriter(
    salida,
    engine="openpyxl"
) as writer:

    nuevo_df.to_excel(writer, index=False)

print("Archivo generado:", salida)

In [ ]:
filas = []

for _, fila in df.iterrows():

    codigos = obtener_codigos(fila["Código presupuestario"])

    # Si no se detectan múltiples códigos, conservar la fila
    if len(codigos) <= 1:
        nueva = fila.copy()
        if len(codigos) == 1:
            nueva["Código presupuestario"] = codigos[0]
        filas.append(nueva)
        continue

    # Primera fila: conserva el monto
    primera = fila.copy()
    primera["Código presupuestario"] = codigos[0]
    filas.append(primera)

    # Filas duplicadas: eliminar el monto
    for codigo in codigos[1:]:

        nueva = fila.copy()
        nueva["Código presupuestario"] = codigo

        # Vaciar el monto de arrastre
        nueva["Monto De arrastre"] = np.nan
        # también sirve:
        # nueva["Monto De arrastre"] = pd.NA

        filas.append(nueva)

nuevo_df = pd.DataFrame(filas)

print("Filas originales :", len(df))
print("Filas nuevas     :", len(nuevo_df))



In [ ]:
nuevo_df["Monto De arrastre"] = nuevo_df["Monto De arrastre"].where(
    pd.notna(nuevo_df["Monto De arrastre"]), ""
)

salida = "plan_de_compras_nuevo.xlsx"

nuevo_df.to_excel(salida, index=False)

print("Archivo generado:", salida)